### Model training and selection with Cross-validation

The model uses elevation, slope, aspect_sin, and aspect_cos. Each time, one spatial fold is used for testing and the other four folds are used for training.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans

In [ ]:
CSV_PATH = Path("/Users/liwei/Desktop/training_matrix_with_absences.csv")
df = pd.read_csv(CSV_PATH)
df.head()

Convert the annular slope direction into two available features of the model; otherwise, it is impossible to distinguish between 1 and 359.<br>
When sin is positive, the slope is more eastward. Negative: The slope is more westward.<br>
When cos is positive, the slope is more northerly. Negative: The slope is further south.<br>

In [ ]:
aspect_rad = np.deg2rad(df["aspect"])
df["aspect_sin"] = np.sin(aspect_rad)
df["aspect_cos"] = np.cos(aspect_rad)
df.head()

1. Use Kmean to group the folds: locations with close coordinate positions are usually more likely to have similar geographical environments and topographic features, such as elevation, slope, and aspect. At the same time, the situation of night parrots may also be similar <br>
2. Another point is that if grouping is done solely based on coordinates, there might be cases where some folds do not have presence or background information, which would prevent the final analysis of the model's performance.<br>
3. Thats's why firstly using presence_sites for classification and find the center points, and then place background_sites into the center points that are close to that center points. <br>
<br>
fit: Find the center point<br>
predict: allocate site into the group of the nearest center point<br>
fit_predict: combine fit and predict<br>

In [ ]:
# 1. Separate unique presence and background locations
presence_sites = df[df["presence"] == 1].drop_duplicates(
    subset=["x_coord", "y_coord"]
)[["x_coord", "y_coord", "presence"]].copy()

background_sites = df[df["presence"] == 0].drop_duplicates(
    subset=["x_coord", "y_coord"]
)[["x_coord", "y_coord", "presence"]].copy()

In [ ]:
presence_sites.head()

In [ ]:
background_sites.head()

In [ ]:
# 2. Create five spatial folds based on presence locations
kmeans = KMeans(n_clusters=5, n_init=50, random_state=1234)

presence_sites["fold"] = kmeans.fit_predict(
    presence_sites[["x_coord", "y_coord"]]
)

# 3. Assign every background location to its nearest presence-based fold
background_sites["fold"] = kmeans.predict(
    background_sites[["x_coord", "y_coord"]]
)

# 4. Combine the two location tables and check fold
unique_sites = pd.concat(
    [presence_sites, background_sites],
    ignore_index=True
)

fold_check = pd.crosstab(
    unique_sites["fold"],
    unique_sites["presence"]
)

fold_check.columns = ["Background (0)", "Presence (1)"]

print(fold_check)

In [ ]:
df = pd.merge(
    df,
    unique_sites[["x_coord", "y_coord", "fold"]],
    on=["x_coord", "y_coord"],
    how="left"
)

print(pd.crosstab(df["fold"], df["presence"]))

## Hyperparameter selection with spatial ROC-AUC

Each candidate is tested using the same five spatial folds. The candidate with the highest Mean ROC-AUC is selected.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import elapid as ela

FEATURES = ["elevation", "slope", "aspect_sin", "aspect_cos"]


### Logistic Regression: choose C

C represent the strength of regularization, smaller C value means a stronger regulariztion and simpler model<br>
Logistic Regression: C ≈ 1 / regularization strength

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10]
mean_auc_scores = []

for C_value in C_values:
    fold_auc_scores = []

    for test_fold in [0, 1, 2, 3, 4]:
        train_df = df[df["fold"] != test_fold]
        test_df = df[df["fold"] == test_fold]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logistic", LogisticRegression(C=C_value, max_iter=1000, random_state=1234))
        ])
        model.fit(train_df[FEATURES], train_df["presence"])

        test_score = model.predict_proba(test_df[FEATURES])[:, 1]
        fold_auc_scores.append(roc_auc_score(test_df["presence"], test_score))

    mean_auc = np.mean(fold_auc_scores)
    mean_auc_scores.append(mean_auc)
    print(f"C={C_value}: Mean ROC-AUC = {mean_auc:.3f}")

best_C = C_values[np.argmax(mean_auc_scores)]
print(f"\nSelected C: {best_C}")

plt.figure(figsize=(7, 4))
plt.plot(C_values, mean_auc_scores, marker="o")
plt.xscale("log")
plt.xticks(C_values, C_values)
plt.ylim(0, 1)
plt.xlabel("C")
plt.ylabel("Mean ROC-AUC")
plt.title("Logistic Regression: C vs ROC-AUC")
plt.show()


### Random Forest: choose max_depth

max_depth refers to the maximum depth that each decision tree is allowed to grow to.<br>
max_depth large → The tree is deep and the rules are complex.<br>
max_depth small → The tree is shallow and the rules are simple.

In [ ]:
max_depth_values = [1, 4, 8, 12, 16, 20, 24]
mean_auc_scores = []

for max_depth in max_depth_values:
    fold_auc_scores = []

    for test_fold in [0, 1, 2, 3, 4]:
        train_df = df[df["fold"] != test_fold]
        test_df = df[df["fold"] == test_fold]

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            random_state=1234
        )
        model.fit(train_df[FEATURES], train_df["presence"])

        test_score = model.predict_proba(test_df[FEATURES])[:, 1]
        fold_auc_scores.append(roc_auc_score(test_df["presence"], test_score))

    mean_auc = np.mean(fold_auc_scores)
    mean_auc_scores.append(mean_auc)
    print(f"max_depth={max_depth}: Mean ROC-AUC = {mean_auc:.3f}")

best_max_depth = max_depth_values[np.argmax(mean_auc_scores)]
print(f"\nSelected max_depth: {best_max_depth}")

plt.figure(figsize=(7, 4))
plt.plot(max_depth_values, mean_auc_scores, marker="o")
plt.ylim(0, 1)
plt.xlabel("max_depth")
plt.ylabel("Mean ROC-AUC")
plt.title("Random Forest: max_depth vs ROC-AUC")
plt.show()


### MaxEnt: choose beta_multiplier

Also, beta_multiplier ≈ Regularization, but the form is different to logistic regression 

In [ ]:
beta_values = [0.5, 1, 2, 3, 4, 5, 6, 7, 10, 20, 30]
mean_auc_scores = []

for beta_value in beta_values:
    fold_auc_scores = []

    for test_fold in [0, 1, 2, 3, 4]:
        train_df = df[df["fold"] != test_fold]
        test_df = df[df["fold"] == test_fold]

        model = ela.MaxentModel(
            transform="cloglog",
            beta_multiplier=beta_value,
            random_state=1234
        )
        model.fit(train_df[FEATURES].copy(), train_df["presence"])

        test_score = model.predict(test_df[FEATURES].copy())
        fold_auc_scores.append(roc_auc_score(test_df["presence"], test_score))

    mean_auc = np.mean(fold_auc_scores)
    mean_auc_scores.append(mean_auc)
    print(f"beta_multiplier={beta_value}: Mean ROC-AUC = {mean_auc:.3f}")

best_beta_multiplier = beta_values[np.argmax(mean_auc_scores)]
print(f"\nSelected beta_multiplier: {best_beta_multiplier}")

plt.figure(figsize=(7, 4))
plt.plot(beta_values, mean_auc_scores, marker="o")
plt.xscale("log")
plt.xticks(beta_values, beta_values)
plt.ylim(0, 1)
plt.xlabel("beta_multiplier")
plt.ylabel("Mean ROC-AUC")
plt.title("MaxEnt: beta_multiplier vs ROC-AUC")
plt.show()


## Final  model

In [ ]:
import joblib

final_maxent_model = ela.MaxentModel(
    transform="cloglog",
    beta_multiplier=6,
    random_state=1234
)
final_maxent_model.fit(df[FEATURES].copy(), df["presence"])

MODEL_PATH = "/Users/liwei/Desktop/WildDiscover/Models/night_parrot_maxent_beta_6.joblib"
joblib.dump(final_maxent_model, MODEL_PATH)